# Example 12.6.5 / 2: Solve a sampled chance constraint

Companion notebook for **12.6.5 Chance Constraints and Risk** in the AMPL lab Documentation, section "Solve a sampled chance constraint". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a sampled chance constraint

The steel plan must fit the available hours with probability at least 0.8 over five weighted
scenarios. A binary `Violate[s]` releases scenario `s` through a row-specific big-M, and
`ViolationMass` limits the violated probability to `epsilon`. After the solve, the run freezes the plan and
counts misses on ten held-out hour values.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                               # steel products
set SCENARIOS;                          # sampled hour scenarios
param probability {SCENARIOS} >= 0;     # scenario weight
param rate {PROD} > 0;                  # tons per hour
param market {PROD} >= 0;               # tons
param profit {PROD} >= 0;               # dollars per ton
param avail {SCENARIOS} >= 0;           # hours in each scenario
param epsilon >= 0 <= 1;                # allowed violation probability

# Row-specific big-M: the most hours any feasible plan can use.
param max_hours = sum {p in PROD} market[p] / rate[p];
# hours over avail[s] that any plan can need
param big_m {s in SCENARIOS} default max(max_hours - avail[s], 0);

var Make {p in PROD} >= 0 <= market[p]; # tons produced, one plan for all scenarios
var Violate {SCENARIOS} binary;         # 1 if the plan may exceed avail[s]

maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];  # dollars

# Hours fit scenario s unless that scenario is counted as violated.
subject to TimeUnlessViolated {s in SCENARIOS}:
    sum {p in PROD} Make[p] / rate[p] <= avail[s] + big_m[s] * Violate[s];

# Violated probability mass, not the number of violated rows.
subject to ViolationMass:
    sum {s in SCENARIOS} probability[s] * Violate[s] <= epsilon;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["SCENARIOS"] = ["s1", "s2", "s3", "s4", "s5"]
ampl.param["probability"] = {"s1": 0.05, "s2": 0.05, "s3": 0.1, "s4": 0.4, "s5": 0.4}
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["avail"] = {"s1": 30, "s2": 34, "s3": 38, "s4": 40, "s5": 44}
ampl.param["epsilon"] = 0.2

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["TotalProfit"].value())
print(ampl.var["Make"].to_dict())
print(ampl.var["Violate"].to_dict())

# Freeze the plan and evaluate it on held-out hours.
hours = ampl.get_value("sum {p in PROD} Make[p] / rate[p]")
held_out = [33, 41, 45, 39, 42, 40, 37, 44, 43, 46]
misses = [round(hours - a, 2) for a in held_out if hours > a + 1e-9]
print(hours, len(misses) / len(held_out), misses)

solved 192000.0
{'bands': 6000, 'coils': 1400}
{'s1': 1, 's2': 1, 's3': 1, 's4': 0, 's5': 0}
40 0.3 [7, 1, 3]


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set SCENARIOS := s1 s2 s3 s4 s5;
param probability := s1 0.05  s2 0.05  s3 0.1  s4 0.4  s5 0.4;
param:   rate  market  profit :=
  bands   200    6000      25
  coils   140    4000      30;
param avail := s1 30  s2 34  s3 38  s4 40  s5 44;
param epsilon := 0.2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display TotalProfit;
display Make;
display Violate;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 branching nodes
TotalProfit = 192000

Make [*] :=
bands  6000
coils  1400
;

Violate [*] :=
s1  1
s2  1
s3  1
s4  0
s5  0
;



### Another way in amplpy

The plan releases `s1`, `s2` and `s3` (mass `0.05 + 0.05 + 0.1 = 0.2`) and uses 40 hours. On the
held-out values it misses 3 of 10 (rate 0.3, above the training 0.2), by 7, 1 and 3 hours.

Two common mistakes change this plan. With the same model.mod and data.py, the run below first sets
every `big_m` to 5, then restores the defaults with `reset data big_m` and replaces the mass limit by
a limit on the number of violated rows (20% of 5 rows is 1):

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["SCENARIOS"] = ["s1", "s2", "s3", "s4", "s5"]
ampl.param["probability"] = {"s1": 0.05, "s2": 0.05, "s3": 0.1, "s4": 0.4, "s5": 0.4}
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["avail"] = {"s1": 30, "s2": 34, "s3": 38, "s4": 40, "s5": 44}
ampl.param["epsilon"] = 0.2

In [11]:
# run.py
ampl.param["big_m"] = {s: 5 for s in ampl.set["SCENARIOS"].to_list()}
ampl.solve(solver="highs", verbose=False)
print("M = 5:", ampl.obj["TotalProfit"].value())

ampl.eval("reset data big_m;")  # back to the row-specific default values
ampl.con["ViolationMass"].drop()
ampl.eval("subject to ViolationCount: sum {s in SCENARIOS} Violate[s] <= 1;")
ampl.solve(solver="highs", verbose=False)
print("count <= 1:", ampl.obj["TotalProfit"].value(), ampl.var["Violate"].to_dict())

M = 5: 171000.0
count <= 1: 166800.0 {'s1': 1, 's2': 0, 's3': 0, 's4': 0, 's5': 0}
